# Prompt Engineering Exercise
### Zero-shot, Few-shot & Chain-of-Thought Prompting

**Instructions:** Work through each section in order. Run the setup cell first. Fill in every cell marked `# TODO`, then answer the reflection questions in the markdown cells provided. Write your answers directly in this notebook.

## Setup

Run this cell once before starting. It installs and loads the model we'll use for all exercises.

In [1]:
!pip install -q transformers

from transformers import pipeline
generator = pipeline('text-generation', model='gpt2')

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

---
## Part 1 — Zero-shot Prompting

**Concept:** In zero-shot prompting, you give the model an instruction with **no examples** and rely on its pre-trained knowledge to produce the answer.

**Task 1.1:** Write a zero-shot prompt asking the model to classify the sentiment of the review below as Positive or Negative.

> Review: "The service was slow and the staff were rude."


In [9]:
# TODO: write a zero-shot prompt for sentiment classification
prompt_zero = """Classify the sentiment of the following review as Positive or Negative.
Review: "The service was slow and the staff were rude."
Sentiment:"""

result = generator(prompt_zero, max_new_tokens=10, num_return_sequences=1)
print(result[0]['generated_text'])

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=10) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Classify the sentiment of the following review as Positive or Negative.

Review: "The service was slow and the staff were rude."

Sentiment: "The service was slow but they could not help


**Task 1.2:** Try a second zero-shot task of your own choice (e.g. translation, summarization, or classifying a different sentence). Write the prompt and run it.

In [10]:
# TODO: write your own zero-shot prompt for a task of your choice
prompt_zero_2 = """Translate the following English sentence into French.
Sentence: "Good morning, how are you?"
Translation:"""

result = generator(prompt_zero_2, max_new_tokens=15, num_return_sequences=1)
print(result[0]['generated_text'])

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=15) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Translate the following English sentence into French.
Sentence: "Good morning, how are you?"
Translation: "Good morning, I'm here."
Thank you, a little bit


**Reflection 1:** In your own words, when would you use zero-shot prompting instead of giving examples? Write 2–3 sentences below.

*Your answer:*

I would use zero-shot prompting when the task is simple and the model already understands what I want from a clear instruction. It saves time because I don't need to provide examples. If the task is straightforward, zero-shot prompting is usually enough to get a good response.



---
## Part 2 — Few-shot Prompting

**Concept:** In few-shot prompting, you give the model a few worked examples before your real question, so it can pick up the pattern (this is called **in-context learning** — no fine-tuning happens).

**Task 2.1:** Below is an incomplete few-shot prompt turning adjectives into nouns. Complete the pattern by adding **two more** adjective/noun example pairs before the final unfinished one, then run it.

In [14]:
# TODO: add two more Adjective/Noun example pairs, keeping the same format
prompt_few = """
Adjective: happy, Noun: happiness
Adjective: kind, Noun: kindness
Adjective: dark, Noun: darkness
Adjective: weak, Noun: weakness
Adjective: sad, Noun:"""

result = generator(prompt_few, max_new_tokens=5, num_return_sequences=1)
print(result[0]['generated_text'])

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=5) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Adjective: happy, Noun: happiness
Adjective: kind, Noun: kindness
Adjective: dark, Noun: darkness
Adjective: weak, Noun: weakness
Adjective: sad, Noun: sad
Adjective


**Task 2.2:** Design your own few-shot prompt for a *different* pattern (for example: country → capital, word → opposite, or singular → plural). Include at least 3 examples before the final unfinished one.

In [15]:
# TODO: write your own few-shot prompt with at least 3 examples
prompt_few_2 = """
Country: France
Capital: Paris
Country: Japan
Capital: Tokyo
Country: India
Capital: New Delhi
Country: Australia
Capital:
"""

result = generator(prompt_few_2, max_new_tokens=5, num_return_sequences=1)
print(result[0]['generated_text'])

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=5) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Country: France
Capital: Paris
Country: Japan
Capital: Tokyo
Country: India
Capital: New Delhi
Country: Australia
Capital:

Germany

Country


**Reflection 2:** Compare your zero-shot and few-shot results. Did giving examples change the quality or format of the output? Explain in 2–3 sentences.

*Your answer:*


The few-shot prompt produced a more structured response because it had examples to follow, while the zero-shot prompt relied only on the instruction. Even though the model did not always generate the expected answer, the examples helped guide its output and made the intended pattern clearer.


---
## Part 3 — Chain-of-Thought (CoT) Prompting

**Concept:** CoT prompting asks the model to reason **step by step** instead of jumping straight to an answer. This is especially useful for math and logic problems.

**Task 3.1:** Below is a word problem. First, run it with a **direct** prompt (no step-by-step instruction) and observe the answer.

In [16]:
question = "A store had 24 candies. It sold 9 and then received a new box of 15. How many candies does the store have now?"

# Direct (non-CoT) prompt
prompt_direct = f"Question: {question}\nAnswer:"

result_direct = generator(prompt_direct, max_new_tokens=30, num_return_sequences=1)
print(result_direct[0]['generated_text'])

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: A store had 24 candies. It sold 9 and then received a new box of 15. How many candies does the store have now?
Answer: The company used to have a store in the area. A new store is being built here in this area of the city.
Posted by: Tom


**Task 3.2:** Now rewrite the prompt to trigger Chain-of-Thought reasoning by adding a phrase like *"Let's think step by step."* Run it and compare the output to Task 3.1.

In [17]:
# TODO: build a Chain-of-Thought version of the prompt
prompt_cot = f"""Question: {question}
Let's think step by step before giving the final answer.
Answer:"""

result_cot = generator(prompt_cot, max_new_tokens=60, num_return_sequences=1)
print(result_cot[0]['generated_text'])

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: A store had 24 candies. It sold 9 and then received a new box of 15. How many candies does the store have now?
Let's think step by step before giving the final answer.
Answer: It's a lot, but it's still worth taking a look.
First, let's look at the number of candies in the store. I've counted them all in one place:
Store Name 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18


**Task 3.3:** Write your own multi-step word problem (addition, subtraction, or multiplication) and test it with both a direct prompt and a CoT prompt.

In [18]:
# TODO: your own word problem
my_question = "Sarah had 18 books. She gave 5 books to her friend and then bought 12 more books. How many books does Sarah have now?"

# Direct prompt
prompt_my_direct = f"Question: {my_question}\nAnswer:"
result = generator(prompt_my_direct, max_new_tokens=30, num_return_sequences=1)
print("DIRECT:\n", result[0]['generated_text'])

# CoT prompt
prompt_my_cot = f"Question: {my_question}\nAnswer: Let's think step by step."
result = generator(prompt_my_cot, max_new_tokens=60, num_return_sequences=1)
print("\nCHAIN-OF-THOUGHT:\n", result[0]['generated_text'])

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


DIRECT:
 Question: Sarah had 18 books. She gave 5 books to her friend and then bought 12 more books. How many books does Sarah have now?
Answer: I know, I know, I know. I don't know. I don't know.
Q: What do you mean by that?


CHAIN-OF-THOUGHT:
 Question: Sarah had 18 books. She gave 5 books to her friend and then bought 12 more books. How many books does Sarah have now?
Answer: Let's think step by step. First, Sarah bought the first book. Then she bought a second book. Then she bought a third book. Then she bought a fourth book. Then she bought a fifth book. Then she bought a sixth book. Then she bought a seventh book. Then she bought a eighth book. Then she bought


**Reflection 3:** Did the CoT prompt produce more accurate or more logically structured reasoning than the direct prompt? Note: GPT-2 is a small, older model, so CoT may not always fix its mistakes — describe what you actually observed.

*Your answer:*


In my experiment, the Chain-of-Thought prompt produced a more structured response because it encouraged the model to reason step by step. However, since GPT-2 is a small and older model, it still did not always generate the correct answer and sometimes produced unrelated text. The CoT prompt improved the format of the response, but it did not completely fix the model's mistakes.


---
## Part 4 — Wrap-up Questions

Answer briefly in your own words:

1. What is the key difference between zero-shot and few-shot prompting?
2. Why is few-shot prompting sometimes called "in-context learning" rather than training?
3. When is Chain-of-Thought prompting most useful, and why?
4. Based on your experiments, what is one limitation of GPT-2 you noticed when generating answers?

*Your answers:*

**1.** Zero-shot prompting gives the model only an instruction, while few-shot prompting includes a few examples to show the model the expected pattern before asking it to complete the task.

**2.** It is called in-context learning because the model learns from the examples provided in the prompt during that conversation. The model's parameters are not updated, so no actual training takes place.

**3.** Chain-of-Thought prompting is most useful for problems that require reasoning, such as math, logic, or multi-step questions. It helps the model break the problem into smaller steps before giving the final answer.

**4.** During my experiments, I noticed that GPT-2 sometimes generated unrelated or incorrect text instead of following the prompt. It was not always able to understand instructions or perform reasoning accurately.

